# 11. KVAE: отдельный визуальный tokenizer

KVAE существует в Kandinsky Lab, но выбранный T2V distill checkpoint использует HunyuanVideo VAE. Совпадение 16 каналов и compression 4×8×8 не делает разные латентные пространства совместимыми.


### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Два смысла слова tokenizer

Qwen tokenizer выдаёт дискретные номера подслов. KVAE visual tokenizer выдаёт непрерывный тензор latent. Здесь речь о втором. KVAE-Image 1.0 использует spatial 8×8, KVAE-Video 1.0 — temporal/spatial 4×8×8.


In [ ]:
import json
for model in ['KVAE-2D-1.0','KVAE-3D-1.0']:
    path=ROOT/'reference/hf'/('kandinskylab--'+model)/'metadata.json'
    metadata=json.loads(path.read_text())
    print(model,'revision:',metadata['sha'])
    print('files:',[x['rfilename'] for x in metadata['siblings'] if x['rfilename'].endswith(('.json','.safetensors','.py'))])


### 2. Почему shape compatibility недостаточно

Два энкодера могут одинаково сжимать изображение, но кодировать смысл разными координатами. Даже перестановка каналов ломает старый decoder. Маленькая линейная аналогия показывает этот эффект.


In [ ]:
image=torch.tensor([1.,2.,3.]); encoder=torch.eye(3); decoder=torch.eye(3)
permutation=torch.tensor([[0.,1.,0.],[0.,0.,1.],[1.,0.,0.]])
z1=encoder@image; z2=permutation@image
print('correct:',decoder@z1,'wrong decoder:',decoder@z2,'matched decoder:',permutation.T@z2)


### 3. Что сообщает настоящая конфигурация

У KVAE encoder/decoder задают ch=128, z_channels=16 и temporal_compress_times=4. Causal блоки используют caches при сегментированной обработке. Нельзя чередовать независимые видео на одном instance: состояния относятся к одному stream.


In [ ]:
cfg=json.loads((ROOT/'reference/hf/kandinskylab--KVAE-3D-1.0/config.json').read_text())
print(json.dumps(cfg,indent=2,ensure_ascii=False))


### 4. Настоящие классы KVAE без больших весов

Pinned KVAE source уже сохранён в reference/kvae. Загружаем только video/image modules в отдельном namespace: audio stack не импортируется. Создадим уменьшенный KVAEVideo с теми же CachedEncoder3D/CachedDecoder3D. Веса случайные: исследуем формы и segmented caches, не качество.


In [ ]:
from labkit.kvae import load_kvae_classes
KVAEVideo,KVAEImage=load_kvae_classes()
small_cfg={'model':{'encoder_params':{'ch':32,'ch_mult':[1,1,1,1],'num_res_blocks':1,'z_channels':4,'temporal_compress_times':4},'decoder_params':{'ch':32,'ch_mult':[1,1,1,1],'num_res_blocks':1,'z_channels':4,'temporal_compress_times':4}}}
small=KVAEVideo(small_cfg).eval()
video=torch.randn(1,3,9,32,32)
with torch.no_grad():
    latent=small.encode(video,seg_len=8).latent_dist.mode()
    restored=small.decode(latent,seg_len=8)
print('small KVAE:',video.shape,'→',latent.shape,'→',restored.shape)
print('parameters:',sum(p.numel() for p in small.parameters()))
del small; release()


### 5. Mode reconstruction с настоящими весами

Синтетическое видео движущегося квадрата позволяет увидеть temporal artifacts. Encode возвращает posterior; decode KVAE возвращает tensor, не DecoderOutput.sample. PSNR считаем на [0,1]. Здесь sample_mode выключает стохастический шум posterior.


In [ ]:
RUN_KVAE=False
if RUN_KVAE:
    from labkit.kvae import load_kvae_classes
    KVAEVideo,KVAEImage=load_kvae_classes()
    from labkit.components import REVISIONS
    device='cuda' if torch.cuda.is_available() else 'cpu'
    dtype=torch.bfloat16 if device=='cuda' else torch.float32
    from huggingface_hub import snapshot_download
    repo='kandinskylab/KVAE-3D-1.0'
    checkpoint=snapshot_download(repo,revision=REVISIONS[repo],allow_patterns=['*.json','*.safetensors','*.bin'])
    model=KVAEVideo.from_pretrained(checkpoint).eval().to(device=device,dtype=dtype)
    video=torch.full((1,3,9,64,64),-1.,device=device,dtype=dtype)
    for t in range(9): video[:,:,t,20:40,5+t*3:21+t*3]=1
    with torch.no_grad():
        latent=model.encode(video,seg_len=16).latent_dist.mode()
        reconstruction=model.decode(latent,seg_len=16).clip(-1,1)
    print('latent:',latent.shape,'reconstruction:',reconstruction.shape)
    mse=((reconstruction.float()-video.float())/2).square().mean().item()
    print('PSNR:',-10*np.log10(max(mse,1e-12)))
    plt.imshow(((reconstruction[0,:,4].float().cpu()+1)/2).permute(1,2,0)); plt.show()
    del model; release()
else: print('Настоящая реконструкция KVAE НЕ выполнялась.')


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
